[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heitorramos/icd/blob/main/exemplos/26-pca-svm/notebook-colab.ipynb)


In [ ]:
# Preparação automática para execução no Google Colab.
# Fora do Colab, esta célula não altera o diretório de trabalho.
try:
    import google.colab  # type: ignore
except ImportError:
    pass
else:
    import os
    import subprocess
    from pathlib import Path

    repository = Path("/content/icd")
    if not repository.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/heitorramos/icd.git", str(repository)
        ], check=True)
    os.chdir(repository / "exemplos/26-pca-svm")
    print("Material preparado em:", Path.cwd())


# PCA aplicado à classificação

Material de apoio — Aula 26

## Objetivos

Este guia usa PCA como transformação de representação e compara dois
classificadores já estudados: regressão logística e KNN. O foco é
aplicado: compreender o papel de cada etapa, montar pipelines sem
vazamento, comparar custo e desempenho e diagnosticar erros.

## Como estudar este capítulo

PCA e classificação resolvem problemas diferentes. PCA cria uma
representação que concentra variabilidade sem consultar os rótulos. A
regressão logística aprende probabilidades por uma relação global; o KNN
decide pela composição de uma vizinhança local. PCA não é obrigatório
para nenhum deles, e maior variância explicada não significa
necessariamente maior poder de classificação.

O exemplo de imagens torna concreta a ideia de representação. Cada pixel
começa como um atributo; o PCA combina pixels em componentes e permite
reconstruções aproximadas. Em seguida, comparamos um classificador
usando os atributos originais e outro usando componentes, observando
desempenho, custo e padrões de erro.

Leia o capítulo como uma comparação controlada. A divisão dos dados e as
métricas permanecem as mesmas; o que muda é a representação entregue ao
classificador. Isso permite discutir quando a redução dimensional
preserva informação útil, quando descarta sinais discriminativos e por
que toda transformação aprendida deve permanecer dentro da pipeline de
validação.

## Base de dados de apoio

Usaremos
[Fashion-MNIST](https://www.kaggle.com/datasets/zalando-research/fashionmnist),
disponível no Kaggle e mantido originalmente pela Zalando Research. Cada
observação é uma imagem $28\times28$ em tons de cinza, associada a uma
de dez classes.

In [ ]:
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(20260827)
data = np.load(Path("data/fashion_mnist_sample.npz"))
images, labels = data["images"], data["labels"]
class_names = np.array(["Camiseta", "Calça", "Suéter", "Vestido", "Casaco",
                        "Sandália", "Camisa", "Tênis", "Bolsa", "Bota"])
pd.Series({"imagens": len(images), "altura": images.shape[1],
           "largura": images.shape[2], "classes": len(np.unique(labels))})

## Análise descritiva

In [ ]:
pd.Series(labels).value_counts().sort_index().rename(index=dict(enumerate(class_names))).to_frame("n")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for label, ax in enumerate(axes.flat):
    image = images[np.where(labels == label)[0][0]]
    ax.imshow(image, cmap="gray")
    ax.set_title(class_names[label])
    ax.axis("off")
plt.tight_layout(); plt.show()

> **Interpretação**
>
> A amostra local tem 600 imagens por classe, portanto acurácia não será
> dominada por uma classe majoritária. As classes de roupas superiores
> possuem contornos parecidos; bolsa, calça e alguns calçados são
> visualmente mais distintos.

## Do gato à aproximação de baixo posto

Uma imagem em tons de cinza é uma matriz $A\in\mathbb R^{m\times n}$. Se

$$A=U\Sigma V^T,$$

a aproximação de posto $k$ é

$$A_k=U_k\Sigma_kV_k^T=\sum_{j=1}^k\sigma_ju_jv_j^T.$$

$u_j$ e $v_j$ descrevem padrões verticais e horizontais; $\sigma_j$ mede
a importância do padrão $j$.

In [ ]:
cat = Image.open(Path("../../slides/assets/aula26-pca-svm/cat-original.jpeg")).convert("L")
cat.thumbnail((420, 280))
A = np.asarray(cat, dtype=float)/255
U, singular_values, Vt = np.linalg.svd(A, full_matrices=False)

fig, axes = plt.subplots(1, 5, figsize=(13, 3))
axes[0].imshow(A, cmap="gray", vmin=0, vmax=1); axes[0].set_title("Original")
for ax, k in zip(axes[1:], [5, 20, 60, 120]):
    Ak = (U[:, :k]*singular_values[:k]) @ Vt[:k]
    ax.imshow(Ak, cmap="gray", vmin=0, vmax=1); ax.set_title(f"posto {k}")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

> **Interpretação**
>
> O posto 5 mantém apenas grandes regiões de luminosidade. Com 60 ou 120
> termos, pelos e bordas reaparecem. A aproximação troca armazenamento e
> detalhe; o melhor $k$ depende do uso da imagem.

## Taxa de armazenamento

A imagem original guarda $mn$ números. A aproximação guarda $mk+k+kn$,
portanto

$$r_k=\frac{k(m+n+1)}{mn}.$$

In [ ]:
m, n = A.shape
pd.DataFrame({
    "posto": [5, 20, 60, 120],
    "fração_armazenada": [k*(m+n+1)/(m*n) for k in [5,20,60,120]],
    "energia_preservada": [np.sum(singular_values[:k]**2)/np.sum(singular_values**2)
                            for k in [5,20,60,120]],
}).round(4)

> **Interpretação**
>
> Energia preservada e qualidade perceptual não são sinônimos. Além
> disso, formatos reais usam codificação, quantização e metadados; a
> razão acima compara apenas números armazenados na matriz e nos fatores
> de baixo posto.

## PCA como representação de observações

Vetorizamos cada roupa em 784 pixels:

$$x_i\in\mathbb R^{784}.$$

Para a matriz centrada $X_c$, escrevemos

$$X_c=U\Sigma V^T,\qquad Z=X_cV_k.$$

$V_k$ contém as $k$ direções principais; $Z$ contém os escores das
observações.

In [ ]:
X = images.reshape(len(images), -1).astype(float)/255
permutation = rng.permutation(len(X))
train, test = permutation[:4800], permutation[4800:]
X_train, X_test = X[train], X[test]
y_train, y_test = labels[train], labels[test]
pca = PCA(n_components=150, svd_solver="randomized", random_state=7).fit(X_train)
pd.Series({"X_treino": X_train.shape, "componentes": pca.components_.shape})

## Variância explicada

$$\operatorname{PVE}_j=\frac{\sigma_j^2}{\sum_{\ell=1}^{r}\sigma_\ell^2}.$$

In [ ]:
cumulative = np.cumsum(pca.explained_variance_ratio_)
pd.Series({
    "componentes_para_80%": int(np.searchsorted(cumulative, .80)+1),
    "componentes_para_90%": int(np.searchsorted(cumulative, .90)+1),
    "componentes_para_95%": int(np.searchsorted(cumulative, .95)+1),
})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(1, len(cumulative)+1), cumulative)
ax.axhline(.90, color="darkorange", linestyle="--")
ax.set(xlabel="componentes", ylabel="variância explicada acumulada")
plt.tight_layout(); plt.show()

> **Interpretação**
>
> PCA preserva variação de $X$, não informação sobre a classe. O número
> de componentes necessário para classificação deve ser validado com o
> classificador, mesmo quando a curva de variância fornece um bom ponto
> de partida.

## Projeção exploratória

In [ ]:
pca2 = PCA(n_components=2, random_state=7)
Z2 = pca2.fit_transform(X_train)
sample = rng.choice(len(Z2), 2200, replace=False)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(Z2[sample, 0], Z2[sample, 1], c=y_train[sample],
           cmap="tab10", s=8, alpha=.5)
ax.set(xlabel="PC1", ylabel="PC2")
plt.tight_layout(); plt.show()

> **Interpretação**
>
> Duas componentes revelam grupos, mas várias classes se sobrepõem. Isso
> não implica que elas sejam inseparáveis no espaço com 50 componentes.
> Projeções bidimensionais são ferramentas exploratórias, não
> estimativas diretas da acurácia possível.

## Duas regras de classificação conhecidas

Depois do PCA, a observação $x_i\in\mathbb R^{784}$ passa a ser
representada pelo escore $z_i\in\mathbb R^k$. A redução muda os
atributos entregues ao modelo, mas não determina como a classe será
escolhida. Compararemos uma regra global e paramétrica com uma regra
local baseada em distância.

Na regressão logística binária,

$$P(Y=1\mid Z=z)=\frac{1}{1+e^{-(\beta_0+\beta^Tz)}}.$$

$z$ é o vetor de componentes da imagem; $\beta_0$ é o intercepto;
$\beta$ contém os coeficientes estimados. Com limiar $0{,}5$, a
fronteira satisfaz $\beta_0+\beta^Tz=0$. No caso multiclasse, o modelo
estima uma probabilidade para cada classe e escolhe a maior.

No KNN, seja $\mathcal N_K(z)$ o conjunto dos índices dos $K$ exemplos
de treino mais próximos de $z$. A previsão é

$$\widehat y(z)=\operatorname{moda}\{y_i:i\in\mathcal N_K(z)\}.$$

$K$ é o número de vizinhos e $y_i$ é o rótulo do vizinho $i$. Um $K$
pequeno cria decisões mais locais e sensíveis; um $K$ grande suaviza a
fronteira.

> **Interpretação**
>
> A regressão logística resume o treino em coeficientes e prevê
> rapidamente. O KNN praticamente apenas armazena o treino, mas precisa
> calcular distâncias ao prever. PCA pode alterar tanto a qualidade
> dessas decisões quanto seu custo computacional.

## Visualizando as duas fronteiras

Para tornar a comparação visível, restringimos temporariamente o
problema a camiseta versus camisa e usamos somente PC1 e PC2.

In [ ]:
mask = np.isin(y_train, [0, 6])
Z_binary = Z2[mask]
y_binary = (y_train[mask] == 6).astype(int)

logistic_2d = LogisticRegression(
    solver="saga", max_iter=300, tol=.01, random_state=7
).fit(Z_binary, y_binary)
knn_2d = KNeighborsClassifier(n_neighbors=15, weights="distance").fit(
    Z_binary, y_binary
)

x_grid = np.linspace(Z_binary[:, 0].min(), Z_binary[:, 0].max(), 220)
y_grid = np.linspace(Z_binary[:, 1].min(), Z_binary[:, 1].max(), 220)
XX, YY = np.meshgrid(x_grid, y_grid)
grid_points = np.c_[XX.ravel(), YY.ravel()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharex=True, sharey=True)
for ax, model, title in [
    (axes[0], logistic_2d, "Logística: fronteira global"),
    (axes[1], knn_2d, "KNN: fronteira local"),
]:
    probability = model.predict_proba(grid_points)[:, 1].reshape(XX.shape)
    ax.contourf(XX, YY, probability, levels=np.linspace(0, 1, 11),
                cmap="RdBu", alpha=.28)
    ax.contour(XX, YY, probability, levels=[.5], colors="black")
    ax.scatter(Z_binary[:, 0], Z_binary[:, 1], c=y_binary,
               cmap="coolwarm", s=10, alpha=.45)
    ax.set(title=title, xlabel="PC1", ylabel="PC2")
plt.tight_layout(); plt.show()

> **Interpretação**
>
> A fronteira logística é uma reta porque o logito é linear nos dois
> componentes. O KNN acompanha irregularidades locais. Ainda assim,
> ambos erram na região de sobreposição: limitar a representação a duas
> componentes descartou informação que nenhum classificador consegue
> reconstruir.

## Distâncias após o PCA

O KNN usa, neste exemplo, a distância euclidiana:

$$d(z_i,z)=\sqrt{\sum_{j=1}^{k}(z_{ij}-z_j)^2}.$$

$z_{ij}$ é o escore da imagem de treino $i$ na componente $j$, e $z_j$ é
o escore da nova imagem nessa componente. As primeiras componentes
possuem maior variância e, portanto, tendem a contribuir mais para essa
distância. Padronizar os escores daria peso semelhante a todas as
componentes e representaria uma escolha substantivamente diferente.

## Pipelines PCA + classificação

In [ ]:
def build_pipeline(n_components, model_name, value):
    if model_name == "Logística":
        classifier = LogisticRegression(
            C=value, solver="saga", max_iter=300, tol=.01, random_state=7
        )
    else:
        classifier = KNeighborsClassifier(n_neighbors=value, weights="distance")
    return Pipeline([
        ("pca", PCA(n_components=n_components,
                    svd_solver="randomized", random_state=7)),
        ("modelo", classifier),
    ])

results = []
fitted = {}
for model_name, value in [("Logística", 1), ("KNN", 7)]:
    for k in [20, 50, 100, 150]:
        model = build_pipeline(k, model_name, value)
        start = perf_counter()
        model.fit(X_train, y_train)
        prediction = model.predict(X_test)
        elapsed = perf_counter() - start
        results.append((model_name, k, accuracy_score(y_test, prediction), elapsed))
        fitted[(model_name, k)] = (model, prediction)

    baseline = (LogisticRegression(C=1, solver="saga", max_iter=300,
                                   tol=.01, random_state=7) if model_name == "Logística"
                else KNeighborsClassifier(n_neighbors=7, weights="distance"))
    start = perf_counter()
    baseline.fit(X_train, y_train)
    prediction = baseline.predict(X_test)
    elapsed = perf_counter() - start
    results.append((model_name, 784, accuracy_score(y_test, prediction), elapsed))
    fitted[(model_name, 784)] = (baseline, prediction)

results_df = pd.DataFrame(
    results, columns=["modelo", "dimensões", "acurácia", "tempo_total_s"]
)
results_df.round(4)

> **Interpretação**
>
> O resultado em 784 dimensões é a referência sem PCA. Os demais
> combinam redução e classificação. Compare cada modelo consigo mesmo
> antes de comparar um modelo com o outro: assim distinguimos o efeito
> da representação do efeito da regra de decisão.

## Visualizando desempenho e custo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
xpos = np.arange(5)
xlabels = ["20", "50", "100", "150", "sem PCA\n(784 pixels)"]
for model_name, part in results_df.groupby("modelo"):
    axes[0].plot(xpos, part["acurácia"], marker="o", label=model_name)
    axes[1].plot(xpos, part["tempo_total_s"], marker="o", label=model_name)
axes[0].set(xlabel="representação", ylabel="acurácia")
axes[1].set(xlabel="representação", ylabel="ajuste + previsão (s)")
for ax in axes:
    ax.legend()
    ax.set_xticks(xpos, xlabels)
plt.tight_layout(); plt.show()

> **Interpretação**
>
> O tempo reúne ajuste e previsão. Isso é importante porque o KNN tem
> ajuste muito barato, mas previsão relativamente cara. Nesta amostra
> pequena, o custo de estimar o PCA domina várias pipelines, e o KNN nos
> pixels originais é especialmente rápido graças à implementação
> otimizada. Ganhos computacionais não devem ser presumidos: dependem do
> tamanho da base, do número de previsões e da implementação.

## Validação da pipeline inteira

O PCA deve ser reajustado dentro de cada parte da validação. Ajustá-lo
uma única vez em todo o conjunto de desenvolvimento permitiria que as
observações de validação influenciassem a representação.

In [ ]:
subset = rng.choice(len(X_train), 1200, replace=False)
folds = StratifiedKFold(3, shuffle=True, random_state=7)
grid = []
settings = {
    "Logística": [0.1, 1, 10],
    "KNN": [3, 7, 15],
}
component_grid = [30, 60, 100]
fold_scores = {(name, k, value): [] for name, values in settings.items()
               for k in component_grid for value in values}

for fit, valid in folds.split(X_train[subset], y_train[subset]):
    # Um único PCA de 100 componentes é ajustado no treino desta parte.
    # Seus primeiros k componentes são exatamente a representação PCA_k.
    pca_fold = PCA(n_components=max(component_grid), svd_solver="randomized",
                   iterated_power=2, random_state=7)
    Z_fit = pca_fold.fit_transform(X_train[subset][fit])
    Z_valid = pca_fold.transform(X_train[subset][valid])

    for model_name, values in settings.items():
        for k in component_grid:
            for value in values:
                if model_name == "Logística":
                    classifier = LogisticRegression(
                        C=value, solver="saga", max_iter=300,
                        tol=.01, random_state=7
                    )
                else:
                    classifier = KNeighborsClassifier(
                        n_neighbors=value, weights="distance"
                    )
                classifier.fit(Z_fit[:, :k], y_train[subset][fit])
                score = classifier.score(Z_valid[:, :k], y_train[subset][valid])
                fold_scores[(model_name, k, value)].append(score)

for (model_name, k, value), scores in fold_scores.items():
    grid.append((model_name, k, value, np.mean(scores), np.std(scores)))

cv_results = pd.DataFrame(
    grid, columns=["modelo", "componentes", "hiperparâmetro",
                  "acurácia_média", "desvio"]
)
cv_results.round(4)

> **Interpretação**
>
> Na regressão logística, o hiperparâmetro exibido é $C$, o inverso da
> intensidade de regularização: valores menores regularizam mais. No
> KNN, ele é $K$, o número de vizinhos. A validação escolhe
> simultaneamente a representação e a flexibilidade do modelo.

## Matriz de confusão

In [ ]:
best_row = results_df.loc[results_df["acurácia"].idxmax()]
best_key = (best_row["modelo"], int(best_row["dimensões"]))
prediction = fitted[best_key][1]
cm = confusion_matrix(y_test, prediction, normalize="true")
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, cmap="Blues", vmin=0, vmax=1,
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set(xlabel="previsto", ylabel="verdadeiro")
ax.tick_params(axis="x", rotation=35); ax.tick_params(axis="y", rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
report = classification_report(y_test, prediction, target_names=class_names,
                               output_dict=True, zero_division=0)
pd.DataFrame(report).T.round(3)

> **Interpretação**
>
> Camisa é confundida com outras roupas superiores com maior frequência.
> Acurácia global esconde essa assimetria. O padrão de erros sugere onde
> investigar rótulos, exemplos adicionais ou representações de textura e
> contorno.

## Quando PCA ajuda — e quando não

PCA pode reduzir redundância, ruído, memória e custo. Pode atrapalhar
quando baixa variância contém sinal preditivo, quando interpretação é
central ou quando a projeção torna dados esparsos densos.

Seu valor deve ser demonstrado fora da amostra, não presumido por uma
porcentagem de variância explicada.

## Como escolher entre logística e KNN

Prefira a regressão logística quando probabilidades, uma fronteira
global, previsão rápida ou interpretação dos coeficientes forem
importantes. Considere KNN quando decisões locais fizerem sentido, a
base de treino couber em memória e a latência de previsão for aceitável.

Em alta dimensão, distâncias podem perder contraste: muitos pontos
parecem igualmente distantes. PCA pode ajudar o KNN ao remover direções
redundantes, mas somente se preservar a estrutura local relevante. Para
produção, considere também atualização, armazenamento, latência,
calibração e explicabilidade.

## PCA passo a passo

1.  **Organizar a matriz.** Linhas representam exemplos e colunas
    representam atributos — neste caso, intensidades dos pixels.
2.  **Centralizar.** Subtraímos a média de cada coluna para que PCA
    descreva variações em torno de um perfil médio.
3.  **Encontrar direções principais.** A primeira captura a maior
    variação possível; as seguintes capturam variações novas e
    ortogonais.
4.  **Projetar.** Cada imagem passa a ser descrita por seus escores nas
    componentes.
5.  **Escolher quantas componentes manter.** Variância explicada e
    desempenho da tarefa ajudam nessa decisão.
6.  **Reconstruir quando necessário.** Multiplicamos os escores pelas
    direções para aproximar a imagem original e observar o que foi
    perdido.

PCA não sabe quais são as classes. Ele preserva variação, não
necessariamente a informação mais útil para classificação.

## Regressão logística passo a passo

1.  **Receber a representação.** Pixels ou componentes entram como
    atributos.
2.  **Calcular escores.** Cada classe recebe uma combinação linear dos
    atributos.
3.  **Transformar em probabilidades.** A função logística, ou sua
    extensão multiclasse, produz valores positivos que somam 1.
4.  **Escolher a classe.** A maior probabilidade determina a previsão
    padrão.
5.  **Controlar a regularização.** O parâmetro $C$ deve ser selecionado
    por validação; valores menores impõem maior regularização.
6.  **Avaliar probabilidades e classes.** Acurácia não substitui
    calibração ou análise da matriz de confusão.

## KNN passo a passo

1.  **Receber a representação.** A escala define o significado das
    distâncias.
2.  **Armazenar o treino.** Não há estimação de uma fronteira
    paramétrica global.
3.  **Calcular distâncias.** Para cada nova imagem, buscamos exemplos
    próximos.
4.  **Selecionar $K$ vizinhos.** $K$ pequeno é mais local; $K$ grande
    suaviza.
5.  **Agregar os rótulos.** Usamos votação simples ou ponderada pela
    distância.
6.  **Validar a pipeline.** Componentes e vizinhos são escolhidos sem
    consultar o teste.

## Como interpretar a comparação

Se PCA reduz muito o número de atributos e mantém desempenho semelhante,
pode valer a pena pelo custo e pela estabilidade. Se um modelo sem PCA
funciona melhor, componentes de baixa variância podem conter informação
discriminativa.

A matriz de confusão mostra quais peças de roupa se confundem. Classes
com forma parecida, como camisa e camiseta, tendem a exigir
representações mais detalhadas do que categorias visualmente distintas.

> **Ideia central**
>
> PCA muda a representação; regressão logística e KNN usam essa
> representação de maneiras diferentes. Como uma etapa altera a entrada
> da outra, cada combinação deve ser avaliada como uma única pipeline.

## Síntese

- PCA reorganiza $X$ por direções de variação;
- compressão e discriminação têm objetivos diferentes;
- regressão logística constrói probabilidades por uma relação global;
- KNN decide localmente a partir de distâncias;
- componentes, regularização e número de vizinhos pertencem à seleção da
  pipeline;
- PCA deve ser ajustado dentro da validação;
- tempo, acurácia e erros por classe orientam a escolha aplicada.

## Bibliografia

- James et al., *An Introduction to Statistical Learning*, capítulos 4 e
  12.
- Hastie, Tibshirani e Friedman, *The Elements of Statistical Learning*,
  capítulos 4, 13 e 14.
- Deisenroth, Faisal e Ong, *Mathematics for Machine Learning*,
  capítulos 10 e 12.
- [Fashion-MNIST — Zalando
  Research](https://github.com/zalandoresearch/fashion-mnist).
- [Exemplo do gato na aula de
  ALC](https://heitorramos.github.io/Slides_ALC/Aula11.html#/qual-foi-a-taxa-de-compressão).